# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook walks through loading, exploring, and performing basic analysis on the [FAIR^2 Croissant dataset (doi:10.71728/senscience.vq4a-28xa)](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and details protein abundance, modifications, and sequences from human samples using mass spectrometry.

In [ ]:
# Ensure `mlcroissant` and extra visualization libraries are installed
!pip install mlcroissant seaborn matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset info
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review all available [record sets](https://mlcommons.org/croissant/record-set/) and field information. All referencing will use their `@id` fields as required for reproducible workflows.

In [ ]:
# List all record sets and their fields
print("Listing all available record sets in the dataset:\n")
record_sets = list(metadata.record_sets)
for rec_set in record_sets:
    print(f"RecordSet name: {rec_set.name}")
    print(f"  @id: {rec_set.id}")
    print(f"  Description: {rec_set.description}")
    print("  Fields:")
    for field in rec_set.fields:
        print(f"    Field name: {field.name}")
        print(f"      @id: {field.id}")
        print(f"      Description: {field.description if hasattr(field, 'description') else ''}")
        print(f"      DataType: {field.data_type}")
    print("-"*60)
# For reference, print the list of recordSet @id's
record_set_ids = [rs.id for rs in record_sets]
print("RecordSet @ids:", record_set_ids)

## 3. Data Extraction
Load data from a selected record set into a DataFrame for analysis. We'll identify the main data table via the `@id` printed above. 

In [ ]:
# Assign the main record set's @id (adjust to the main table's @id)
# To help, we display all @ids, names, and suggestions above.
main_record_set_id = record_set_ids[0]   # <-- Adjust this index if another should be primary

print(f"Extracting records from record set @id: {main_record_set_id}")

# You can extract from multiple record sets:
dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"{rec_id}: shape {df.shape}")

# Review columns in the main record set
print("Main record set columns:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Below are common analysis operations: filtering numeric fields, normalization, and grouping by categorical variables.

To proceed, identify a numeric field `@id` (from field listing above)---for illustration, let's use the first field of dtype 'Float', 'Integer', or 'Number'.

In [ ]:
# Automatically select a numeric field @id
from numpy import number

main_record_set = [rs for rs in metadata.record_sets if rs.id == main_record_set_id][0]

# Get first numeric field
numeric_field_id = None
group_field_id = None
for field in main_record_set.fields:
    # Accept fields of dataType Float, Integer or Number
    if str(field.data_type).lower() in ["float", "integer", "number"]:
        numeric_field_id = field.id
        break
# Try to find a string/categorical field as group candidate
for field in main_record_set.fields:
    if str(field.data_type).lower() in ["string", "text"]:
        group_field_id = field.id
        break

print(f"Numeric field @id: {numeric_field_id}")
print(f"Grouping field @id: {group_field_id}")

# Filter > threshold (median for demonstration)
df = dataframes[main_record_set_id]
if numeric_field_id is not None and numeric_field_id in df.columns:
    # Ensure numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Z-normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by categorical/string field if available
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped (mean) by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
We'll display histograms, boxplots, or simple relationships if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
We have successfully loaded and explored the FAIR^2 Croissant dataset of mass spectrometry analysis for extracellular vesicles. This notebook demonstrated how to discover record sets, extract data using `mlcroissant`, perform basic filtering and grouping, and visualize main numeric attributes. For domain-specific analysis (e.g., protein enrichment, annotation comparison), continue by referencing `@id`s and field metadata for reproducibility.